In [0]:
df_hospital_a = spark.read.parquet("/mnt/bronze/hospital-a/transactions")

In [0]:
df_hospital_b = spark.read.parquet("/mnt/bronze/hospital-b/transactions")

In [0]:
df_merged = df_hospital_a.unionByName(df_hospital_b)

In [0]:
df_merged.createOrReplaceTempView('transactions')

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW quality_checks AS
SELECT 
concat(transactionid,'-',datasource) AS transactionid,
transactionid as src_transactionid,
encounterid,
patientid,
providerid,
deptid,
visitdate,
servicedate,
paiddate,
visittype,
amount,
amounttype,
paidamount,
claimid,
payorid,
procedurecode,
icdcode,
lineofbusiness,
medicaidid,
medicareid,
insertdate as src_insertdate,
modifieddate as src_modifieddate,
datasource,
CASE 
  WHEN encounterid IS NULL OR patientid IS NULL OR transactionid IS NULL OR visitdate IS NULL THEN TRUE
  ELSE FALSE
END AS is_quarantined
FROM transactions

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver

In [0]:
%sql 
CREATE TABLE IF NOT EXISTS silver.transactions(
  transactionid STRING,
  scr_transactionid STRING,
  encounterid STRING,
  patientid STRING,
  providerid STRING,
  deptid STRING,
  visitdate DATE,
  servicedate DATE,
  paiddate DATE,
  visittype STRING,
  amount DOUBLE,
  amounttype STRING,
  paidamount DOUBLE,
  claimid STRING,
  payorid STRING,
  procedurecode INT,
  icdcode STRING,
  lineofbusiness STRING,
  medicaidid STRING,
  medicareid STRING,
  src_insertdate DATE,
  src_modifieddate DATE,
  datasource STRING,
  is_quarantined BOOLEAN,
  audit_insert_date TIMESTAMP,
  audit_modified_date TIMESTAMP,
  is_current BOOLEAN
)
USING DELTA

In [0]:
%sql 
MERGE INTO silver.transactions AS target 
USING quality_checks AS source
ON target.transactionid = source.transactionid
AND target.is_current = TRUE
WHEN MATCHED 
AND(
  target.scr_transactionid != source.src_transactionid
  OR target.encounterid != source.encounterid
  OR target.patientid != source.patientid
  OR target.providerid != source.providerid
  OR target.deptid != source.deptid
  OR target.visitdate != source.visitdate
  OR target.servicedate != source.servicedate
  OR target.paiddate != source.paiddate
  OR target.visittype != source.visittype
  OR target.amount != source.amount
  OR target.amounttype != source.amounttype
  OR target.paidamount != source.paidamount
  OR target.claimid != source.claimid
  OR target.payorid != source.payorid
  OR target.procedurecode != source.procedurecode
  OR target.icdcode != source.icdcode
  OR target.lineofbusiness != source.lineofbusiness
  OR target.medicaidid != source.medicaidid
  OR target.medicareid != source.medicareid
  OR target.src_insertdate != source.src_insertdate
  OR target.src_modifieddate != source.src_modifieddate
  OR target.datasource != source.datasource
  OR target.is_quarantined != source.is_quarantined
)
THEN 
UPDATE SET 
  target.is_current = FALSE,
  target.audit_modified_date = current_timestamp()


In [0]:
%sql
MERGE INTO silver.transactions AS target
USING quality_checks AS source
ON target.transactionid = source.transactionid
AND target.is_current = true
WHEN NOT MATCHED 
THEN INSERT (
  transactionid,
  scr_transactionid,
  encounterid,
  patientid,
  providerid,
  deptid,
  visitdate,
  servicedate,
  paiddate,
  visittype,
  amount,
  amounttype,
  paidamount,
  claimid,
  payorid,
  procedurecode,
  icdcode,
  lineofbusiness,
  medicaidid,
  medicareid,
  src_insertdate,
  src_modifieddate,
  datasource,
  is_quarantined,
  audit_insert_date,
  audit_modified_date,
  is_current
)
VALUES(
  source.transactionid,
  source.src_transactionid,
  source.encounterid,
  source.patientid,
  source.providerid,
  source.deptid,
  source.visitdate,
  source.servicedate,
  source.paiddate,
  source.visittype,
  source.amount,
  source.amounttype,
  source.paidamount,
  source.claimid,
  source.payorid,
  source.procedurecode,
  source.icdcode,
  source.lineofbusiness,
  source.medicaidid,
  source.medicareid,
  source.src_insertdate,
  source.src_modifieddate,
  source.datasource,
  source.is_quarantined,
  current_timestamp(),
  current_timestamp(),
  TRUE
)


In [0]:
%sql
select * from silver.transactions LIMIT 10